# ATML PA2 - Task 5 (RLVR vs RLAIF) on Kaggle (2x T4)

This notebook only **orchestrates** the repository scripts (`task5_feedback/`). No training: SFT and the
supplied frozen RLVR / RLAIF adapters are evaluated with greedy decoding, the exact verifier and the fixed
pairwise AI judge (Qwen2.5-3B-Instruct, 4-bit). No input dataset is needed; the course assets are downloaded.

**Before running:** Accelerator = *GPU T4 x2*, Internet = *On*.

| GPU 0 | GPU 1 |
|---|---|
| Step 1: GSM8K (300 prompts) x 3 policies + pairwise judge | Step 3: SVAMP (100) x 3 policies + judge, then Step 2: diagnostic set |

Output: `task5_artifacts.zip` (results/task5_feedback + report/task5).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"
BRANCH = "main"
WORKDIR = "/tmp/repo"   # outside /kaggle/working: keeps checkpoints out of the notebook output

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
else:  # an existing clone (e.g. from an earlier run) is synced to the latest pushed commit
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run Task 5
Resumable within a session: finished generations and cached judge calls are reused. Per-GPU logs: `results/task5_feedback/logs/`.

In [ ]:
!bash task5_feedback/run_all.sh

## 5. Quick look at the results

In [ ]:
from IPython.display import Markdown, Image, display
for t in ["in_domain", "transfer_drop", "verifier_judge_agreement", "diagnostic_pairs", "diagnostic_sensitivity", "feedback_cost"]:
    p = f"report/task5/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["diagnostic_preferences", "in_vs_out_of_domain", "failure_types"]:
    p = f"report/task5/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Package the artifacts
Unzip `task5_artifacts.zip` at the repository root locally. `python -m task5_feedback.compare_feedback` rebuilds the report material on CPU.

In [ ]:
!cd /tmp/repo && zip -qr /kaggle/working/task5_artifacts.zip results/task5_feedback report/task5
!ls -lh /kaggle/working/